In [ ]:
import pandas as pd

teams = [
    ("Algeria", "ALG"),
    ("Argentina", "ARG"),
    ("Australia", "AUS"),
    ("Austria", "AUT"),
    ("Belgium", "BEL"),
    ("Bosnia and Herzegovina", "BIH"),
    ("Brazil", "BRA"),
    ("Cabo Verde", "CPV"),
    ("Canada", "CAN"),
    ("Colombia", "COL"),
    ("Congo DR", "COD"),
    ("Côte d'Ivoire", "CIV"),
    ("Croatia", "CRO"),
    ("Curaçao", "CUW"),
    ("Czechia", "CZE"),
    ("Ecuador", "ECU"),
    ("Egypt", "EGY"),
    ("England", "ENG"),
    ("France", "FRA"),
    ("Germany", "GER"),
    ("Ghana", "GHA"),
    ("Haiti", "HAI"),
    ("Iraq", "IRQ"),
    ("IR Iran", "IRN"),
    ("Japan", "JPN"),
    ("Jordan", "JOR"),
    ("Korea Republic", "KOR"),
    ("Mexico", "MEX"),
    ("Morocco", "MAR"),
    ("Netherlands", "NED"),
    ("New Zealand", "NZL"),
    ("Norway", "NOR"),
    ("Panama", "PAN"),
    ("Paraguay", "PAR"),
    ("Portugal", "POR"),
    ("Qatar", "QAT"),
    ("Saudi Arabia", "KSA"),
    ("Scotland", "SCO"),
    ("Senegal", "SEN"),
    ("South Africa", "RSA"),
    ("Spain", "ESP"),
    ("Sweden", "SWE"),
    ("Switzerland", "SUI"),
    ("Tunisia", "TUN"),
    ("Türkiye", "TUR"),
    ("USA", "USA"),
    ("Uruguay", "URU"),
    ("Uzbekistan", "UZB")
]

teams_df = pd.DataFrame(
    {
        "team_id": range(1, len(teams) + 1),
        "team_name": [t[0] for t in teams],
        "team_code": [t[1] for t in teams],
    }
)

teams_df.to_csv("teams.csv", index=False)

print(teams_df)
print(f"\nSaved teams.csv with {len(teams_df)} teams.")

    team_id               team_name team_code
0         1                 Algeria       ALG
1         2               Argentina       ARG
2         3               Australia       AUS
3         4                 Austria       AUT
4         5                 Belgium       BEL
5         6  Bosnia and Herzegovina       BIH
6         7                  Brazil       BRA
7         8              Cabo Verde       CPV
8         9                  Canada       CAN
9        10                Colombia       COL
10       11                Congo DR       COD
11       12           Côte d'Ivoire       CIV
12       13                 Croatia       CRO
13       14                 Curaçao       CUW
14       15                 Czechia       CZE
15       16                 Ecuador       ECU
16       17                   Egypt       EGY
17       18                 England       ENG
18       19                  France       FRA
19       20                 Germany       GER
20       21                   Ghan

In [ ]:
# ==========================================
# PART 1
# Setup + Upload
# ==========================================

import os
import re
import shutil
import pandas as pd

from google.colab import files

print("Upload all five FIFA team statistics text files:")
uploaded = files.upload()

Upload all five FIFA team statistics text files:


Saving team stats attacking.txt to team stats attacking.txt
Saving team stats defending.txt to team stats defending.txt
Saving team stats discipline.txt to team stats discipline.txt
Saving team stats distribution.txt to team stats distribution.txt
Saving team stats goalkeeping.txt to team stats goalkeeping.txt


In [ ]:
# ==========================================
# PART 2
# Helper Functions + Parser
# ==========================================

def normalize_column(col):
    col = col.lower().strip()
    col = col.replace("%", "percent")
    col = col.replace("/", "_")
    col = col.replace("-", "_")
    col = col.replace("(", "")
    col = col.replace(")", "")
    col = col.replace(":", "")
    col = col.replace(".", "")
    col = re.sub(r"\s+", "_", col)
    col = re.sub(r"_+", "_", col)
    return col


def read_lines(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        return [x.strip().replace("\ufeff", "") for x in f.readlines()]


def locate_file(expected_name):
    expected = expected_name.lower()

    for f in uploaded.keys():
        if f.lower() == expected:
            return f

    for f in uploaded.keys():
        if expected.replace(".txt", "") in f.lower():
            return f

    raise FileNotFoundError(expected_name)


def parse_team_file(filepath):

    raw = read_lines(filepath)

    header = None
    start = None

    for i, line in enumerate(raw):
        if line.lower().startswith("rank"):
            header = [normalize_column(x) for x in line.split(",")]
            start = i + 1
            break

    if header is None:
        raise Exception("Header not found.")

    expected_stats = len(header) - 2

    tokens = [x for x in raw[start:] if x != ""]

    rows = []
    i = 0

    while i < len(tokens):

        if not tokens[i].isdigit():
            i += 1
            continue

        rank = tokens[i]
        i += 1

        if i >= len(tokens):
            break

        team = tokens[i]
        i += 1

        stats = []

        while i < len(tokens) and len(stats) < expected_stats:
            stats.append(tokens[i])
            i += 1

        if len(stats) != expected_stats:
            continue

        rows.append([rank, team] + stats)

    return pd.DataFrame(rows, columns=header)

In [ ]:
# ==========================================
# PART 3
# Parse All Team Statistics Files
# ==========================================

FILES = {
    "attacking": "team stats attacking.txt",
    "distribution": "team stats distribution.txt",
    "defending": "team stats defending.txt",
    "discipline": "team stats discipline.txt",
    "goalkeeping": "team stats goalkeeping.txt",
}

dfs = {}

for name, filename in FILES.items():
    try:
        actual_file = locate_file(filename)
        df = parse_team_file(actual_file)
        dfs[name] = df

        print(f"{name:<15}: {len(df)} rows parsed")

    except Exception as e:
        print(f"{name:<15}: ERROR -> {e}")

print("\nParsed DataFrames\n")

for name, df in dfs.items():
    print("=" * 70)
    print(name.upper())
    display(df.head())

attacking      : 48 rows parsed
distribution   : 48 rows parsed
defending      : 48 rows parsed
discipline     : 48 rows parsed
goalkeeping    : 48 rows parsed

Parsed DataFrames

ATTACKING


,rank,team,goals,assists,attempts_at_goal,attempt_on_target,off_target_attempts,attempts_at_goal_conv_rate,attempts_inside_the_penalty_area,attempts_outside_the_penalty_area,headed_attempts_at_goal,xg,xg_efficiency,corners,possession_control
0,1,Argentina,19,12,113,46,48,17,65,48,22,15.38,1.24x,37,55
1,2,France,16,14,120,50,51,13,65,55,10,13.79,1.15x,48,52
2,3,England,14,11,99,41,35,14,73,26,25,12.7,1.1x,36,50
3,3,Belgium,14,10,112,35,43,13,74,38,16,10.62,1.32x,24,46
4,5,Spain,13,9,120,42,53,11,71,49,17,14.96,0.87x,45,58


DISTRIBUTION


,rank,team,distribution_passes,passes_completed,passing_accuracy,crosses,crossing_acuracy,take_ons_completed,defensive_linebreaks_attempted,defensie_linebreaks_accuracy,switches_of_play_attempted,switches_of_play_acc
0,1,Argentina,4772,4324,91,102,33,50,122,60,34,94
1,2,Spain,4592,4156,91,154,21,61,175,65,40,95
2,3,France,3865,3452,89,121,16,80,151,64,36,92
3,4,Morocco,3665,3240,88,99,24,66,110,52,26,85
4,5,England,3479,3098,89,162,26,70,182,65,43,79


DEFENDING


,rank,team,own_goals,goals_conceded,forced_turnover,ball_recovery_time,defensive_pressures_applied,defensive_pressures_directly_applied
0,1,Qatar,2,10,101,60.48,832,122
1,1,Egypt,2,7,224,82.81,1360,238
2,3,Uzbekistan,1,11,135,50,944,142
3,3,Iraq,1,12,122,61.26,851,155
4,3,Switzerland,1,6,285,84.65,1581,281


DISCIPLINE


,rank,teams,fouls_against,fouls_for,yellow_cards,red_cards,indirect_red_cards,offsides
0,1,Argentina,88,79,9,0,0,21
1,2,Switzerland,87,89,8,1,1,13
2,3,Spain,80,66,6,0,0,17
3,4,Belgium,78,68,6,1,0,8
4,5,Morocco,74,88,7,0,0,9


GOALKEEPING


,rank,teams,clean_sheets,goals_conceded,goalkeeper_saves,goalkeeper_actions_inside_the_penalty_area,goalkeeper_actions_outside_the_penalty_area
0,1,Spain,6,1,14,108,84
1,2,France,4,4,11,80,64
2,2,Colombia,4,1,7,57,54
3,2,Mexico,4,3,8,60,49
4,5,Argentina,2,7,9,87,73


In [ ]:
# ==========================================
# PART 4
# Normalize + Export CSVs
# ==========================================

import os

# Load master teams table
teams_df = pd.read_csv("teams.csv")

# Create team name -> team_id lookup
team_lookup = dict(zip(teams_df["team_name"], teams_df["team_id"]))

# Create output folder
os.makedirs("output", exist_ok=True)

OUTPUT_FILES = {
    "attacking": "team_attacking.csv",
    "distribution": "team_distribution.csv",
    "defending": "team_defending.csv",
    "discipline": "team_discipline.csv",
    "goalkeeping": "team_goalkeeping.csv",
}

for name, df in dfs.items():

    temp = df.copy()

    # Some files use 'teams' instead of 'team'
    if "teams" in temp.columns:
        temp.rename(columns={"teams": "team"}, inplace=True)

    # Remove extra spaces
    temp["team"] = temp["team"].str.strip()

    # Map to team_id
    temp["team_id"] = temp["team"].map(team_lookup)

    # Check for unmatched teams
    unmatched = temp[temp["team_id"].isna()]

    if not unmatched.empty:
        print(f"\n{name.upper()} - Unmatched teams:")
        print(unmatched["team"].unique())

    # Drop team name
    temp.drop(columns=["team"], inplace=True)

    # Place team_id after rank
    cols = temp.columns.tolist()
    cols.remove("team_id")
    cols.insert(1, "team_id")
    temp = temp[cols]

    # Save CSV
    output_path = os.path.join("output", OUTPUT_FILES[name])
    temp.to_csv(output_path, index=False)

    print(f"✓ {OUTPUT_FILES[name]} saved ({len(temp)} rows)")

print("\nAll files exported successfully!")

✓ team_attacking.csv saved (48 rows)
✓ team_distribution.csv saved (48 rows)
✓ team_defending.csv saved (48 rows)
✓ team_discipline.csv saved (48 rows)
✓ team_goalkeeping.csv saved (48 rows)

All files exported successfully!


In [19]:
import shutil
from google.colab import files

shutil.make_archive("fifa_normalized_teams_dataset", "zip", "output")
files.download("fifa_normalized_teams_dataset.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>